# 10장 · RAG(검색 증강 생성) 실습

RAG(Retrieval-Augmented Generation)는 LLM이 답변을 생성하기 전에 외부 문서에서 관련 정보를 먼저 **검색(Retrieval)** 해 가져오고, 그 문서를 질문과 함께 모델에 **증강(Augmentation)** 으로 전달한 뒤, 이를 근거로 **생성(Generation)** 하는 기술이다.

이렇게 하면 모델이 학습하지 않은 정보에도 답할 수 있고, 환각(hallucination)이 줄어들며, 답변의 출처도 알려 줄 수 있다.

이 노트북은 다음 순서로 진행한다.
1. PDF 문서 읽기 (PyPDFLoader)
2. 청킹 — 긴 텍스트를 청크 단위로 자르기 (RecursiveCharacterTextSplitter)
3. 오버랩 처리 - 텍스트를 청크로 나눌 때 앞뒤 조각이 일부 내용을 겹치게 만드는 것
4. 임베딩 — 텍스트를 벡터로 변환 (OpenAIEmbeddings)
5. 벡터 DB — 크로마 DB 생성/저장
6. 리트리버(질문과 관련있는 문서 조각을 찾는 역할)로 유사 청크 검색
7. 청크를 근거로 답변 생성 (create_stuff_documents_chain)
8. 질의 확장 (query augmentation)

## 0. 패키지 설치

필요한 패키지를 설치한다. (이미 설치돼 있으면 건너뛴다)

In [1]:
%pip install PyMuPDF pypdf langchain langchain_community langchain_chroma langchain_openai python-dotenv streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. PyPDFLoader로 문서 읽기

`data/2040_seoul_plan.pdf` 파일을 읽어 텍스트 데이터를 추출한다.

> 실습 데이터는 아래에서 내려받아 `data/` 폴더에 넣는다.
> - 서울: https://urban.seoul.go.kr/view/html/PMNU5020400001?booksType=BK0300
> - 뉴욕: https://a860-gpp.nyc.gov/concern/parent/gx41mm584/file_sets/1z40kw69m

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일을 읽어서 텍스트 데이터를 추출한다.
loader = PyPDFLoader('./data/2040_seoul_plan.pdf')
data_seoul = loader.load()
print(len(data_seoul)) # 로드된 문서의 개수를 출력한다.
print(data_seoul[0]) # 로드된 문서의 첫 번째 페이지 내용을 출력한다.

C:\Users\kim10\AppData\Local\Temp\ipykernel_22300\783101057.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\kim10\Desktop\AI-Boot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


205
page_content='' metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2020 11.0.0.5178', 'creationdate': '2024-12-12T18:16:11+09:00', 'author': 'SI', 'moddate': '2024-12-12T18:16:11+09:00', 'pdfversion': '1.4', 'source': './data/2040_seoul_plan.pdf', 'total_pages': 205, 'page': 0, 'page_label': '1'}


## 2. 청킹 — 긴 텍스트를 청크 단위로 자르기

`RecursiveCharacterTextSplitter`로 텍스트를 1000자 단위로 나누고, 청크 사이 맥락이 끊기지 않도록 overlap을 100자로 둔다.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100) # 텍스트 청크 단위 정의
all_splits = text_splitter.split_documents(data_seoul) # 텍스트를 청크단위로 분할한다.
print(f"청크 개수: {len(all_splits)}") # 청크의 개수를 출력한다.

청크 개수: 308


In [4]:
# 앞쪽 청크 몇 개만 출력해 본다.
for i, split in enumerate(all_splits[:3]): # 앞쪽 3개의 청크만 출력
    print(f"Split {i+1}:------------------------------------\n")
    print(split) # 청크 내용을 출력 (현재 청크가 커서 오버랩이 적용되지 않은 것 같아 보임.)

Split 1:------------------------------------

page_content='「2040 서울도시기본계획」을 발간하며
지난 3년간 코로나19 팬데믹으로 전 세계가 심각한 타격을 받아왔지만, 대한민국의 수도 서울은 혁신적인 디지털 기술과 뛰어난 시민 의식, 풍부한 자연환경을 토대로 도시의 가능성과 잠재력을 확인할 수 있었습니다. 「2040 서울도시기본계획」은 기후위기, 디지털 전환, 생활양식의 변화 등 글로벌 대도시가 당면한 과제에 대한 해법을 제시하고 있습니다.첫째, 보행일상권으로의 공간구조 개편입니다. 서울을 하나의 기준으로 관리하던 지금까지의 방식에서 벗어나, 미래의 서울은 다양한 지역특성을 반영한 차별화된 계획체계를 통해 동네단위의 자족적 생활권으로 재구성하게 됩니다. 이는 기후위기, 팬데믹 등 각종 재난상황에서도 도시활동이 가능한 공간단위로, 서울 어디에서나 시민 삶의 질이 보장되는 새로운 차원의 균형발전정책 기반이 될 것입니다.둘째, 과감하고 유연한 도시계획 기조로의 전환입니다. 앞으로의 도시계획은 미래 여건변화에 유연하고 신속하게 대응할 수 있는 체계를 통해 현장에서 강력하게 작동하는 수단이 될 것입니다. 미래지향적인 계획철학을 토대로 융복합적인 토지이용제도를 실현하고 수변녹지와 연계된 생활공간을 조성하는 등 도시계획체계를 과감하고 유연하게 전환했습니다.' metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2020 11.0.0.5178', 'creationdate': '2024-12-12T18:16:11+09:00', 'author': 'SI', 'moddate': '2024-12-12T18:16:11+09:00', 'pdfversion': '1.4', 'source': './data/2040_seoul_plan.pdf', 'total_pages': 205, 'page': 1, 'page_label': '2'}
Split 2:--------------------

In [5]:
# all_splits의 요소는 랭체인의 Document 클래스 인스턴스이다.
print(type(all_splits[0]))

<class 'langchain_core.documents.base.Document'>


## 3. (참고) 오버랩 직접 확인하기

`RecursiveCharacterTextSplitter`가 이미 overlap을 넣어 주지만, 개념 확인을 위해 인접 청크의 앞 100자를 이어 붙여 맥락이 어떻게 겹치는지 눈으로 본다.

> ⚠️ 아래 셀은 원본 청크를 변형하므로 **개념 확인용**이다. 실제 DB 저장 전에는 다시 청킹하거나 이 셀을 건너뛴다.

In [6]:
# 오버랩 처리 전
if len(all_splits) > 51:
    print(all_splits[50].page_content)
    print('----------------------')
    print(all_splits[51].page_content)

32제2장 미래상과 목표
9) 대기질 개선과 폐기물 관리 등 광역거버넌스 차원으로 풀어야 할 과제 발생증가하는 생활폐기물 배출량, 다시 우려되는 대기오염Ÿ서울에서 발생하는 폐기물을 처리해 왔던 인천 수도권매립지의 매립 종료가 2025년 예정되면서 폐기물 처리는 주요한 이슈로 떠올랐다. -2019년 폐기물 배출량은 2015년보다 0.6만 톤 증가한 4.8만 톤이 배출, 이 중 생활폐기물은 9,847톤으로 2015년에 비해 409톤 증가Ÿ이후 갑작스러운 팬데믹으로 인한 온라인 쇼핑, 배달 음식 소비증가 등을 고려하면 폐기물 배출량은 더욱 증가할 것으로 우려되고 있다.
[그림 2-14] 폐기물 발생량 및 1인당 생활폐기물 배출량(2010~2019)자료: 환경부, 서울시 폐기물 재활용 현황 통계, 각 연도Ÿ팬데믹 기간, 미세먼지와 관련한 대기질 지표는 일시적으로 좋아지는 양상을 보였다. -단기간 고농도 발생으로 생활에 직접적인 영향을 주는 주의보 발령 일수는 2019년 14일에서 2020년 4일로 감소하였다가 2021년 16일로 다시 증가Ÿ대기질 개선과 폐기물 감소를 위해서는 서울 대도시권 및 중앙정부 차원에서 대응방안을 마련하고 지속적으로 실행해 나가야 한다.10) 생활서비스 및 공원녹지 수요 증가생활서비스 시설은 확충 중, 노년인구를 위한 시설은 수요 증가에 비해 더딘 공급Ÿ생활서비스 시설의 공급 지표는 모든 분야에서 긍정적인 결과를 보인다. -보육시설, 아동·노인여가·재가노인복지시설, 장애인생활시설과 문화시설, 공공도서관, 공공체육시설, 병원, 공원 등의 시설도 지속적인 확충 중Ÿ2020년 65세 이상 노년인구는 전체 인구의 15.4%로 최근 10년 사이 비율이 6.0%p 늘어나면서 노인복지시설에 대한 수요가 증대되고 있다.
----------------------
제1절 서울의 변화진단33-노인여가복지시설 역시 확충되고 있으나, 노령화의 속도를 따라잡지 못해 2020년에는 노인 천 명당 2.1개소로 2010년에 비해 0.9개소 감소Ÿ다양한 생활서비스 시설의 양

## 4. 임베딩 — 텍스트를 벡터로 변환하기

오픈AI의 임베딩 모델 `text-embedding-3-large`로 텍스트를 벡터로 변환한다. 실행하면 숫자로 이루어진 3072차원 벡터가 출력된다.

> 임베딩/생성 모두 **실제 OpenAI API 키**가 필요하다. `.env`의 `OPENAI_API_KEY`에 정식 키(sk-proj-...)를 넣는다.

In [8]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
v = embedding.embed_query("서울시의 온실가스 저감 정책은 뭐야?")
print(v[:5])          # 앞 5개 값만 미리 보기
print(len(v))         # 벡터 차원 수

[0.006916046142578125, -0.054931640625, -0.014312744140625, -0.0043487548828125, -0.0030460357666015625]
3072


## 5. 벡터 DB — 크로마 DB 만들고 저장하기

청크를 임베딩해 크로마 DB(벡터를 저장하고 검색하는 DB)에 저장한다. `persist_directory` 경로에 sqlite3 파일이 생성된다.
이미 DB가 있으면 새로 만들지 않고 불러온다.

In [9]:
from langchain_chroma import Chroma
import os

# 이 경로에 크로마 DB가 생성되며 sqlite3 파일이 저장된다.
persist_directory = './chroma_store'

if not os.path.exists(persist_directory): # 해당 경로가 존재하지 않으면 새로 만든다.
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents( # 문서들을 벡터화하여 크로마 DB를 생성한다.
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory,
    )
else:
    print("Loading existing Chroma store")
    vectorstore = Chroma( # 기존에 생성된 크로마 DB를 불러온다.
        persist_directory=persist_directory,
        embedding_function=embedding,
    )

Creating new Chroma store


## 6. 리트리버로 유사 청크 가져오기

리트리버는 질문을 벡터로 변환한 뒤, 벡터 DB에서 유사도가 높은 청크 k개를 찾아 가져온다.

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("서울시의 환경 정책에 대해 궁금해")

for d in docs:
    print(d)
    print('------')

page_content='제4절 기후·환경 부문1. 개요Ÿ기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측면과 연관되어 있어 향후 서울시의 적극적인 대응이 필요하다.Ÿ탄소중립 목표뿐만 아니라 미세먼지로부터 시민 건강을 지키기 위해서는 건물, 교통, 에너지 등 도시의 주요 인프라 전반의 혁신이 요구되며, 이를 위해 새로운 기술과 혁신적 제도가 필요하다. 제로에너지 건물, 친환경 차량 및 교통 인프라의 확대, 자원·에너지 순환 기반 조성으로 온실가스와 미세먼지 배출량을 획기적으로 감축해야 한다. Ÿ기후변화에 따른 폭염, 풍수해, 도심열섬현상 등 기후재난 및 극한 기후현상이 심해질 것으로 전망되어 보다 능동적인 대비가 필요하다. Ÿ한편, 환경보존과 쾌적한 도시환경을 위해 도심 곳곳 시민 모두가 누릴 수 있는 도심숲과 생활공원 등 녹색공간을 조성하고, 이를 수변 공간과 연계하여 풍부하고 지속가능한 자연환경이 확보될 수 있도록 한다.Ÿ장기적인 측면에서 시민 개개인과 기업 등 다양한 도시 내 행위자의 적극적인 협조가 필수적이며 이를 위해 중앙정부와 서울시 환경계획 담당부서와의 협력적이고 포용적인 거버넌스 체계를 구축하도록 한다.목표 전략3-12050 탄소중립 실현을 위한 도시 인프라 전환3-1-1건물 부문의 탄소배출을 감축하기 위한 친환경 기술 개발 및 적극 적용3-1-2미래 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라 확충3-1-3에너지 전환을 위한 청정에너지 기반 구축3-1-4대기 환경을 고려한 공간계획과 배출원 관리체계 강화3-2건강한 순환도시 조성을 위한자립적인 자원순환 체계 구축3-2-1자원순환·관리 자립을 위한 분산형 폐기물처리 시설 구축3-2-2기후 행동 포용적 거버넌스 구축을 위한 시민 행동 활성화3-3사람과 자연의 공존을 위한친환경 생태도시 구축3-3-1건물 에너지 분야 효율성 개선 및 도심 속 생물 다양성 확보3-3-2지속가능한 통합 물순환 체계 구축3-4다양한 수변을 경험할 수 있는수변감성도

## 7. 청크를 바탕으로 답변하는 시스템 만들기

`create_stuff_documents_chain`은 검색해 온 청크(context)를 프롬프트에 끼워 넣어 답변을 생성하는 체인을 만든다.

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

### 메시지 저장하고 답변 출력하기

`ChatMessageHistory`에 대화를 저장하면서 답변을 생성한다.

In [16]:
from langchain_community.chat_message_histories import ChatMessageHistory

# 채팅 메시지 저장할 메모리 객체 생성
chat_history = ChatMessageHistory()
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("서울시의 온실가스 저감 정책에 대해 알려줘.")

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer)

print(answer)

서울시는 온실가스 저감을 위한 다양한 정책을 추진하고 있습니다. 주요 내용은 다음과 같습니다:

1. **탄소중립 목표**: 서울시는 2005년 대비 70% 온실가스를 감축하는 목표를 설정하고 이를 실현하기 위한 구체적인 전략을 마련하고 있습니다.

2. **친환경 건축 및 에너지 효율성 개선**: 건물 부문에서의 탄소배출 감축을 위해, 제로에너지 건물 및 에너지 효율화 사업을 추진하고 있습니다. 이를 통해 2026년까지 100만 호의 에너지 효율화 사업을 완료할 계획입니다.

3. **청정 에너지 기반 구축**: 미래의 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라를 확충하여 에너지 전환을 지원하고 있습니다.

4. **자원 순환 체계 구축**: 자원 순환 및 관리의 자립성을 높이기 위해 분산형 폐기물 처리 시설을 구축하고, 시민 참여를 촉진하기 위한 다양한 정책과 캠페인을 운영하고 있습니다.

5. **시민 참여 증진**: 시민의 기후 행동을 촉진하기 위해, 다양한 교육 프로그램을 마련하고 대중교통 이용 및 일회용품 사용 감축을 유도하는 가이드라인을 제공하고 있습니다.

6. **친환경 생태도시 조성**: 도심 내 녹지 공간 확대와 생물 다양성 확보를 위해 토양과 물 관리 체계를 효율적으로 통합하여 자원 순환과 자연공존을 도모하고 있습니다.

이와 같은 다양한 정책과 프로그램을 통해 서울시는 도시의 기후와 환경 문제에 적극적으로 대응하며, 지속 가능한 도시 발전을 목표로 하고 있습니다.


In [17]:
# 대화 기록 출력
for m in chat_history.messages:
    print(m)

content='서울시의 온실가스 저감 정책에 대해 알려줘.' additional_kwargs={} response_metadata={}
content='서울시는 온실가스 저감을 위한 다양한 정책을 추진하고 있습니다. 주요 내용은 다음과 같습니다:\n\n1. **탄소중립 목표**: 서울시는 2005년 대비 70% 온실가스를 감축하는 목표를 설정하고 이를 실현하기 위한 구체적인 전략을 마련하고 있습니다.\n\n2. **친환경 건축 및 에너지 효율성 개선**: 건물 부문에서의 탄소배출 감축을 위해, 제로에너지 건물 및 에너지 효율화 사업을 추진하고 있습니다. 이를 통해 2026년까지 100만 호의 에너지 효율화 사업을 완료할 계획입니다.\n\n3. **청정 에너지 기반 구축**: 미래의 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라를 확충하여 에너지 전환을 지원하고 있습니다.\n\n4. **자원 순환 체계 구축**: 자원 순환 및 관리의 자립성을 높이기 위해 분산형 폐기물 처리 시설을 구축하고, 시민 참여를 촉진하기 위한 다양한 정책과 캠페인을 운영하고 있습니다.\n\n5. **시민 참여 증진**: 시민의 기후 행동을 촉진하기 위해, 다양한 교육 프로그램을 마련하고 대중교통 이용 및 일회용품 사용 감축을 유도하는 가이드라인을 제공하고 있습니다.\n\n6. **친환경 생태도시 조성**: 도심 내 녹지 공간 확대와 생물 다양성 확보를 위해 토양과 물 관리 체계를 효율적으로 통합하여 자원 순환과 자연공존을 도모하고 있습니다.\n\n이와 같은 다양한 정책과 프로그램을 통해 서울시는 도시의 기후와 환경 문제에 적극적으로 대응하며, 지속 가능한 도시 발전을 목표로 하고 있습니다.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


## 8. 질의 확장(query augmentation) 구현하기

"뉴욕은?" 처럼 모호한 질문은 리트리버가 관련 청크를 잘 찾지 못한다.
기존 대화 맥락을 활용해 질문을 **명료한 한 문장**으로 다듬는 것을 질의 확장이라고 한다.

`StrOutputParser`는 모델의 응답을 문자열로 변환해 준다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser  # 문자열 출력 파서

query_for_nyc = "뉴욕은?" # 사용자의 질문이 모호한 경우, 기존 대화 내용을 활용하여 명확한 질문으로 변환

# 기존 대화 내용을 활용해 query augmentation 수행
query_augmentation_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="messages"),  # 기존 대화 내용
        (
            "system",
            "기존의 대화 내용을 활용하여 사용자의 아래 질문의 의도를 파악하여 "
            "명료한 한 문장의 질문으로 변환하라. 대명사나 이, 저, 그와 같은 표현을 "
            "명확한 명사로 바꾸어라.",
        ),
    ]
)

# chat 객체와 프롬프트를 연결하고 결과를 문자열로 변환한다.
query_augmentation_chain = query_augmentation_prompt | chat | StrOutputParser()

In [19]:
# 질문을 더 명확하게 변환하기
augmented_query = query_augmentation_chain.invoke({
    "messages": chat_history.messages,
    "query": query_for_nyc,
})

print(augmented_query)

서울시의 온실가스 저감 정책에 대해 구체적으로 설명해 줄 수 있습니까?


In [ ]:
# 확장된 질문으로 리트리버 실행하고 결과 출력하기
docs = retriever.invoke(augmented_query)

for d in docs:
    print(d)
    print('------')

In [ ]:
# 언어 모델에서 답변 생성하기
chat_history.add_user_message(query_for_nyc)  # query_for_nyc에 "뉴욕은?" 추가

answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer)

print(answer)

## 9. 스트림릿으로 챗봇 완성하기

위 흐름(질의 확장 → 검색 → 답변)을 스트림릿 앱으로 묶은 것이 `retriever.py` + `rag.py` 이다.
이 노트북을 한 번 실행해 `chroma_store/`를 만들어 둔 뒤 아래 명령으로 실행한다.

```powershell
streamlit run chapter10/rag.py
```